# Raahi: export the Android detector (no training)

YOLOE-26 with our **27 classes** baked in → `.tflite` for the phone (+ `.onnx`, `.labels.json`).
Runs in ~5–10 min. **No GPU needed** (CPU runtime is fine). Just *Runtime → Run all*.
At the end a `raahi_models.zip` downloads (Colab) or appears in Output (Kaggle).

Class list generated from `shared/classes.json`; to change it, edit that file and run
`python training/build_export_notebook.py` again.

In [ ]:
%pip install -q -U ultralytics
import os, json, shutil
from pathlib import Path
import ultralytics
print("ultralytics", ultralytics.__version__)
WORK = Path("/kaggle/working" if os.path.exists("/kaggle/working") else "/content")
os.chdir(WORK)
OUT = WORK / "raahi_models"; OUT.mkdir(exist_ok=True)

In [ ]:
# ---- generated from shared/classes.json (don't edit here) ----
PROMPTS = ["person", "chair", "office chair", "table", "desk", "sofa", "couch", "bench", "door", "wooden door", "glass door", "window", "wall", "pillar", "column", "stairs", "staircase", "trash can", "dustbin", "potted plant", "flower pot", "laptop", "computer monitor", "television", "led screen", "whiteboard", "projector screen", "cupboard", "cabinet", "shelf", "bookshelf", "bag", "backpack", "suitcase", "cardboard box", "bottle", "water cooler", "water dispenser", "pedestal fan", "bed"]
PROMPT2ID = {"person": "person", "chair": "chair", "office chair": "chair", "table": "table", "desk": "desk", "sofa": "sofa", "couch": "sofa", "bench": "bench", "door": "door", "wooden door": "door", "glass door": "glass_door", "window": "window", "wall": "wall", "pillar": "pillar", "column": "pillar", "stairs": "stairs", "staircase": "stairs", "trash can": "trash_bin", "dustbin": "trash_bin", "potted plant": "plant_pot", "flower pot": "plant_pot", "laptop": "laptop", "computer monitor": "monitor", "television": "tv_led", "led screen": "tv_led", "whiteboard": "whiteboard", "projector screen": "projector_screen", "cupboard": "cupboard", "cabinet": "cupboard", "shelf": "shelf", "bookshelf": "shelf", "bag": "bag", "backpack": "bag", "suitcase": "bag", "cardboard box": "box", "bottle": "bottle", "water cooler": "water_cooler", "water dispenser": "water_cooler", "pedestal fan": "fan_stand", "bed": "bed"}
MIN_CONF = {"person": 0.4, "chair": 0.35, "table": 0.35, "desk": 0.35, "sofa": 0.35, "bench": 0.35, "door": 0.3, "glass_door": 0.35, "window": 0.35, "wall": 0.5, "pillar": 0.4, "stairs": 0.4, "trash_bin": 0.3, "plant_pot": 0.35, "laptop": 0.35, "monitor": 0.35, "tv_led": 0.35, "whiteboard": 0.35, "projector_screen": 0.4, "cupboard": 0.35, "shelf": 0.35, "bag": 0.35, "box": 0.35, "bottle": 0.35, "water_cooler": 0.35, "fan_stand": 0.35, "bed": 0.35, "child": 0.6}

# ---- what to export: (model size, input size). n@320 = fastest; s@416 = more accurate ----
EXPORTS = [("n", 320), ("n", 416), ("s", 416)]
MAKE_INT8 = True     # extra *_int8.tflite for phones without a usable GPU
print(len(PROMPTS), "prompts ->", len(set(PROMPT2ID.values())), "classes")

In [ ]:
from ultralytics import YOLOE

def fresh(size):
    m = YOLOE(f"yoloe-26{size}-seg.pt")
    m.set_classes(PROMPTS)
    return m

def save(src, dest):
    src = Path(src)
    if src.is_dir():                                  # litert export returns a folder
        src = sorted(src.rglob("*.tflite"))[0]
    shutil.copy(src, dest)
    print("  ->", dest.name, f"{dest.stat().st_size/1e6:.1f} MB")

results = {}
for size, imgsz in EXPORTS:
    stem = f"raahi_yoloe_{size}_{imgsz}"
    print(f"\n=== {stem}")
    m = fresh(size)
    names = [m.names[i] for i in range(len(m.names))]
    (OUT / f"{stem}.labels.json").write_text(json.dumps(
        {"names": names, "class_ids": [PROMPT2ID.get(n) for n in names], "min_conf": MIN_CONF},
        ensure_ascii=False, indent=1), encoding="utf-8")
    save(fresh(size).export(format="litert", imgsz=imgsz), OUT / f"{stem}.tflite")
    save(fresh(size).export(format="onnx", imgsz=imgsz, simplify=True), OUT / f"{stem}.onnx")
    if MAKE_INT8:
        try:
            save(fresh(size).export(format="litert", imgsz=imgsz, quantize=8, data="coco8.yaml"),
                 OUT / f"{stem}_int8.tflite")
        except Exception as e:
            print("  int8 skipped:", e)

## Check: run the exported `.tflite` on a test image (input/output shapes are what the Android plugin expects)

In [ ]:
from ultralytics import YOLO
import numpy as np
try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    import tensorflow as tf; Interpreter = tf.lite.Interpreter

img = "https://ultralytics.com/images/bus.jpg"
for f in sorted(OUT.glob("*.tflite")):
    it = Interpreter(model_path=str(f)); it.allocate_tensors()
    ins = [d["shape"].tolist() for d in it.get_input_details()]
    outs = [d["shape"].tolist() for d in it.get_output_details()]
    print(f"{f.name:32s} in {ins}  out {outs}")
    try:
        r = YOLO(str(f), task="segment").predict(img, imgsz=int(f.stem.split("_")[3]), verbose=False)[0]
        print("    detections:", [(r.names[int(c)], round(float(s), 2)) for c, s in zip(r.boxes.cls, r.boxes.conf)][:6])
    except Exception as e:
        print("    (predict check skipped:", e, ")")

## Download

In [ ]:
shutil.make_archive(str(WORK / "raahi_models"), "zip", OUT)
for f in sorted(OUT.iterdir()): print(f"{f.name:34s} {f.stat().st_size/1e6:6.1f} MB")
try:
    from google.colab import files; files.download(str(WORK / "raahi_models.zip"))
except ImportError:
    print("Kaggle: download raahi_models.zip from the Output panel on the right")